In [1]:
%pip install -U \
  "autogluon.timeseries==1.5.0" \
  "torch>=2.6.0" \
  "torchvision>=0.21.0" \
  "torchaudio>=2.6.0" \
  "transformers>=4.48.0" \
  "tokenizers>=0.21.0" \
  "numpy==1.26.4"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.8/244.8 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 100.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.9/98.9 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 

In [ ]:
import pandas as pd
import numpy as np
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
from sklearn.metrics import r2_score,mean_squared_error
import os

In [ ]:
base_dir = os.getcwd()

csv_path = os.path.join(base_dir, "..", "data", "training", "skopje_final_data.csv")
df = pd.read_csv(csv_path,low_memory=False)
df.head()

,timestamp,sensorId,humidity,pm10,pm25,pressure,temperature,wind_speed,neighbor1_humidity,neighbor2_humidity,...,neighbor5_wind_speed,season,hour_sin,hour_cos,month_sin,month_cos,day_sin,day_cos,is_weekend,is_heating_season
0,2023-12-01 00:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,75.916667,NaN,NaN,981.158046,6.359756,7.391761,74.00,75.916667,...,11.246759,winter,0.000000,1.000000,-0.5,0.866025,-0.433884,-0.900969,0,1
1,2023-12-01 01:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,76.331140,NaN,NaN,981.250000,6.315041,7.238935,73.75,76.331140,...,12.313894,winter,0.258819,0.965926,-0.5,0.866025,-0.433884,-0.900969,0,1
2,2023-12-01 02:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,76.574786,NaN,NaN,981.250000,6.236111,7.570863,74.50,76.574786,...,10.972620,winter,0.500000,0.866025,-0.5,0.866025,-0.433884,-0.900969,0,1
3,2023-12-01 03:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,76.818376,NaN,NaN,981.133333,6.378968,7.031864,75.75,76.818376,...,10.233123,winter,0.707107,0.707107,-0.5,0.866025,-0.433884,-0.900969,0,1
4,2023-12-01 04:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,77.098291,NaN,NaN,981.044444,6.593254,6.751236,73.75,77.098291,...,10.739832,winter,0.866025,0.500000,-0.5,0.866025,-0.433884,-0.900969,0,1


In [3]:
df.describe()

,humidity,pm10,pm25,pressure,temperature,wind_speed,neighbor1_humidity,neighbor2_humidity,neighbor3_humidity,neighbor4_humidity,...,neighbor4_wind_speed,neighbor5_wind_speed,hour_sin,hour_cos,month_sin,month_cos,day_sin,day_cos,is_weekend,is_heating_season
count,1.315275e+06,1.017354e+06,1.014088e+06,1.315275e+06,1.315275e+06,1.315800e+06,1.315275e+06,1.315275e+06,1.315275e+06,1.315275e+06,...,1.315800e+06,1.315800e+06,1.315800e+06,1.315800e+06,1.315800e+06,1.315800e+06,1.315800e+06,1.315800e+06,1.315800e+06,1.315800e+06
mean,5.566136e+01,2.492260e+01,1.387223e+01,9.828889e+02,1.720202e+01,4.959989e+00,5.530974e+01,5.483988e+01,5.550372e+01,5.499049e+01,...,4.848925e+00,4.847676e+00,-1.850473e-17,-5.551014e-17,-2.785087e-03,-3.554140e-03,-2.996776e-03,-6.839945e-04,2.872777e-01,4.145007e-01
std,1.782989e+01,4.210747e+01,2.289562e+01,9.442534e+00,9.319345e+00,3.199850e+00,1.748186e+01,1.797581e+01,1.810821e+01,1.741685e+01,...,2.876215e+00,2.877265e+00,7.071070e-01,7.071070e-01,7.068597e-01,7.073399e-01,7.073425e-01,7.068648e-01,4.524924e-01,4.926358e-01
min,0.000000e+00,0.000000e+00,0.000000e+00,9.220000e+02,-5.800000e+01,0.000000e+00,0.000000e+00,4.500000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,-1.000000e+00,-1.000000e+00,-1.000000e+00,-1.000000e+00,-9.749279e-01,-9.009689e-01,0.000000e+00,0.000000e+00
25%,4.185390e+01,6.000000e+00,3.250000e+00,9.787500e+02,9.703252e+00,2.811690e+00,4.175000e+01,4.100000e+01,4.141880e+01,4.150000e+01,...,2.833955e+00,2.817445e+00,-7.071068e-01,-7.071068e-01,-5.000000e-01,-8.660254e-01,-7.818315e-01,-9.009689e-01,0.000000e+00,0.000000e+00
50%,5.575000e+01,1.200000e+01,6.750000e+00,9.830000e+02,1.641870e+01,4.510787e+00,5.536111e+01,5.450000e+01,5.537326e+01,5.525000e+01,...,4.503598e+00,4.503598e+00,6.123234e-17,-6.123234e-17,0.000000e+00,-1.836970e-16,0.000000e+00,-2.225209e-01,0.000000e+00,0.000000e+00
75%,6.950000e+01,2.766667e+01,1.533333e+01,9.880000e+02,2.432623e+01,6.432324e+00,6.900000e+01,6.850000e+01,6.964757e+01,6.867029e+01,...,6.341009e+00,6.359372e+00,7.071068e-01,7.071068e-01,8.660254e-01,5.000000e-01,7.818315e-01,6.234898e-01,1.000000e+00,1.000000e+00
max,9.900000e+01,1.873750e+03,9.980000e+02,1.195000e+03,5.950000e+01,4.566469e+01,9.900000e+01,9.900000e+01,9.900000e+01,9.900000e+01,...,2.926869e+01,2.926869e+01,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,9.749279e-01,1.000000e+00,1.000000e+00,1.000000e+00


In [4]:
len(df)

1315800

In [5]:
df['sensorId'].value_counts()

,count
sensorId,
sensor_dev_84941_208,17544
007f2b03-94e6-47b3-9e3e-44273354acd5,17544
01cf1cec-bf2d-41b3-8cd5-e8bd720f01b4,17544
sensor_dev_78082_739,17544
sensor_dev_78164_619,17544
...,...
1001,17544
1002,17544
1003,17544


In [6]:
TARGET = 'pm25'
ID_COL = 'sensorId'
TIME_COL = 'timestamp'
PREDICTION_LENGTH = 512

In [7]:
df[TIME_COL] = pd.to_datetime(df[TIME_COL])

In [8]:
df[TIME_COL] = df[TIME_COL].dt.tz_convert(None)

In [9]:
print(df["timestamp"].dtype)

datetime64[ns]


In [10]:
data = TimeSeriesDataFrame.from_data_frame(
    df,
    id_column=ID_COL,
    timestamp_column=TIME_COL
)

In [11]:
data.columns

Index(['humidity', 'pm10', 'pm25', 'pressure', 'temperature', 'wind_speed',
       'neighbor1_humidity', 'neighbor2_humidity', 'neighbor3_humidity',
       'neighbor4_humidity', 'neighbor5_humidity', 'neighbor1_pressure',
       'neighbor2_pressure', 'neighbor3_pressure', 'neighbor4_pressure',
       'neighbor5_pressure', 'neighbor1_temperature', 'neighbor2_temperature',
       'neighbor3_temperature', 'neighbor4_temperature',
       'neighbor5_temperature', 'neighbor1_wind_speed', 'neighbor2_wind_speed',
       'neighbor3_wind_speed', 'neighbor4_wind_speed', 'neighbor5_wind_speed',
       'season', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'day_sin',
       'day_cos', 'is_weekend', 'is_heating_season'],
      dtype='object')

In [12]:
train_data, test_data = data.train_test_split(prediction_length=PREDICTION_LENGTH)

In [13]:
forecast_columns  = [ elem for elem in data.columns if elem not in ['pm10','pm25']]
forecast_columns

['humidity',
 'pressure',
 'temperature',
 'wind_speed',
 'neighbor1_humidity',
 'neighbor2_humidity',
 'neighbor3_humidity',
 'neighbor4_humidity',
 'neighbor5_humidity',
 'neighbor1_pressure',
 'neighbor2_pressure',
 'neighbor3_pressure',
 'neighbor4_pressure',
 'neighbor5_pressure',
 'neighbor1_temperature',
 'neighbor2_temperature',
 'neighbor3_temperature',
 'neighbor4_temperature',
 'neighbor5_temperature',
 'neighbor1_wind_speed',
 'neighbor2_wind_speed',
 'neighbor3_wind_speed',
 'neighbor4_wind_speed',
 'neighbor5_wind_speed',
 'season',
 'hour_sin',
 'hour_cos',
 'month_sin',
 'month_cos',
 'day_sin',
 'day_cos',
 'is_weekend',
 'is_heating_season']

In [14]:
predictor = TimeSeriesPredictor(
    target= TARGET,
    prediction_length=PREDICTION_LENGTH,
    eval_metric="MASE",
    known_covariates_names=forecast_columns
)

In [15]:
predictor.fit(
    train_data,
    presets="chronos2",
    time_limit=300
)

Beginning AutoGluon training... Time limit = 300s
AutoGluon will save models to '/content/AutogluonModels/ag-20260803_110505'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.56/14.56 GB
Total GPU Memory:   Free: 14.56 GB, Allocated: 0.00 GB, Total: 14.56 GB
GPU Count:          1
Memory Avail:       9.46 GB / 12.67 GB (74.6%)
Disk Space Avail:   58.32 GB / 112.64 GB (51.8%)
Setting presets to: chronos2

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': MASE,
 'hyperparameters': {'Chronos2': {'model_path': 'autogluon/chronos-2'}},
 'known_covariates_names': ['humidity',
                            'pressure',
                            'temperature',
                            'wind_speed',
        

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/478M [00:00<?, ?B/s]

	19.27   s     = Training runtime
Training complete. Models trained: ['Chronos2']
Total runtime: 20.01 s
Best model: Chronos2


In [ ]:
forecast_df = pd.read_csv('../data/raw/skopje_forecast_weather.csv')
forecast_df

,timestamp,sensorId,lat,lon,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,surface_pressure
0,2025-11-09 16:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,12.300,89.000000,2.200000,NaN,948.50000
1,2025-11-09 17:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,12.000,90.000000,3.600000,NaN,948.40000
2,2025-11-09 18:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,11.600,92.000000,2.200000,NaN,947.90000
3,2025-11-09 19:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,11.400,94.000000,1.800000,NaN,948.20000
4,2025-11-09 20:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,11.300,95.000000,2.800000,NaN,947.80000
...,...,...,...,...,...,...,...,...,...
501079,2026-03-01 17:00:00+00:00,sensor_dev_8550_72,42.000000,21.380000,5.438,71.722170,1.548418,324.462250,995.15894
501080,2026-03-01 18:00:00+00:00,sensor_dev_8550_72,42.000000,21.380000,7.638,66.715480,1.310420,15.945477,995.11540
501081,2026-03-01 19:00:00+00:00,sensor_dev_8550_72,42.000000,21.380000,6.788,69.456400,2.305125,308.659820,995.31110
501082,2026-03-01 20:00:00+00:00,sensor_dev_8550_72,42.000000,21.380000,6.338,68.128060,1.913426,318.814150,995.45435


In [ ]:
neighbourhood_matrix = pd.read_csv('../data/neighbors_data/skopje_neighbors.csv')
valid_sensors = df['sensorId'].unique()
neighbors_df_clean = neighbourhood_matrix[
    (neighbourhood_matrix['sensor_id'].isin(valid_sensors)) &
    (neighbourhood_matrix['neighbor_id'].isin(valid_sensors))
].copy()

In [18]:
def append_neighbors(df_hourly, neighbors_df, weather_cols, k_search=20, k_keep=5):
    # 1. Standardize the neighbor list
    # Ensure we only take the top K based on distance
    neighbors_topk = (
        neighbors_df.sort_values(["sensor_id", "distance_km"])
        .groupby("sensor_id")
        .head(k_search)
        .copy()
    )

    # Track original distance rank
    neighbors_topk['dist_rank'] = neighbors_topk.groupby("sensor_id").cumcount() + 1

    # 2. Merge with main data
    # We use 'neighbor_id' from the matrix to match 'sensorId' in the hourly data
    neighbor_values = neighbors_topk.merge(
        df_hourly[['sensorId', 'timestamp'] + weather_cols],
        left_on='neighbor_id',
        right_on='sensorId',
        how='inner'
    )

    # 3. Filter for availability
    # The 'sensor_id' here is the ORIGINAL sensor we are finding neighbors for
    available_topk = (
        neighbor_values.sort_values(['sensor_id', 'timestamp', 'dist_rank'])
        .groupby(['sensor_id', 'timestamp'])
        .head(k_keep)
        .copy()
    )

    # Create the 1, 2, 3 rank for the wide-format columns
    available_topk['final_rank'] = available_topk.groupby(['sensor_id', 'timestamp']).cumcount() + 1

    # 4. Pivot to wide format
    pivot_df = available_topk.pivot(
        index=['sensor_id', 'timestamp'],
        columns='final_rank',
        values=weather_cols
    )

    # Clean up column names: neighbor1_temp, neighbor2_temp, etc.
    if isinstance(pivot_df.columns, pd.MultiIndex):
        pivot_df.columns = [f"neighbor{rank}_{col}" for col, rank in pivot_df.columns]
    else:
        # Handle case with only one weather column
        pivot_df.columns = [f"neighbor{i}_{weather_cols[0]}" for i in pivot_df.columns]

    pivot_df = pivot_df.reset_index()

    # 5. Final Join back to original data
    df_result = df_hourly.merge(
        pivot_df,
        left_on=['sensorId', 'timestamp'],
        right_on=['sensor_id', 'timestamp'],
        how='left'
    ).drop(columns=['sensor_id'])

    return df_result


In [19]:
weather_cols = ['humidity', 'pressure', 'temperature', 'wind_speed']

In [20]:
forecast_df.drop(columns='wind_direction_10m',inplace=True)
forecast_df.columns

Index(['timestamp', 'sensorId', 'lat', 'lon', 'temperature_2m',
       'relative_humidity_2m', 'wind_speed_10m', 'surface_pressure'],
      dtype='object')

In [21]:
forecast_df.rename(columns={"temperature_2m":"temperature","relative_humidity_2m":"humidity","surface_pressure":"pressure","wind_speed_10m":"wind_speed"},inplace=True)
forecast_df.columns

Index(['timestamp', 'sensorId', 'lat', 'lon', 'temperature', 'humidity',
       'wind_speed', 'pressure'],
      dtype='object')

In [22]:
forecast_df = append_neighbors(forecast_df,neighbors_df_clean, weather_cols)
forecast_df

,timestamp,sensorId,lat,lon,temperature,humidity,wind_speed,pressure,neighbor1_humidity,neighbor2_humidity,...,neighbor1_temperature,neighbor2_temperature,neighbor3_temperature,neighbor4_temperature,neighbor5_temperature,neighbor1_wind_speed,neighbor2_wind_speed,neighbor3_wind_speed,neighbor4_wind_speed,neighbor5_wind_speed
0,2025-11-09 16:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,12.300,89.000000,2.200000,948.50000,86.0,86.0,...,14.0,14.1,14.1,14.0,13.5,3.0,3.0,3.0,3.0,1.8
1,2025-11-09 17:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,12.000,90.000000,3.600000,948.40000,92.0,92.0,...,13.3,13.4,13.4,13.4,13.1,5.2,5.2,5.2,5.2,1.3
2,2025-11-09 18:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,11.600,92.000000,2.200000,947.90000,95.0,95.0,...,12.9,13.0,13.0,13.0,12.7,4.0,4.0,4.0,4.0,2.2
3,2025-11-09 19:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,11.400,94.000000,1.800000,948.20000,95.0,95.0,...,12.8,12.9,12.9,12.8,12.6,4.4,4.4,4.4,4.4,1.8
4,2025-11-09 20:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,11.300,95.000000,2.800000,947.80000,94.0,94.0,...,12.7,12.8,12.8,12.8,12.5,4.7,4.7,4.7,4.7,0.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501079,2026-03-01 17:00:00+00:00,sensor_dev_8550_72,42.000000,21.380000,5.438,71.722170,1.548418,995.15894,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
501080,2026-03-01 18:00:00+00:00,sensor_dev_8550_72,42.000000,21.380000,7.638,66.715480,1.310420,995.11540,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
501081,2026-03-01 19:00:00+00:00,sensor_dev_8550_72,42.000000,21.380000,6.788,69.456400,2.305125,995.31110,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
501082,2026-03-01 20:00:00+00:00,sensor_dev_8550_72,42.000000,21.380000,6.338,68.128060,1.913426,995.45435,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
def extract_time_features(df, timestamp_col='timestamp'):

    month = df[timestamp_col].dt.month

    df['season'] = np.select(
        [
            month.isin([12, 1, 2]),
            month.isin([3, 4, 5]),
            month.isin([6, 7, 8]),
            month.isin([9, 10, 11])
        ],
        [
            'winter',
            'spring',
            'summer',
            'autumn'
        ]
    )

    df['hour_sin'] = np.sin(2 * np.pi * df[timestamp_col].dt.hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df[timestamp_col].dt.hour / 24)


    df['month_sin'] = np.sin(2 * np.pi * (df[timestamp_col].dt.month - 1) / 12)
    df['month_cos'] = np.cos(2 * np.pi * (df[timestamp_col].dt.month - 1) / 12)

    df['day_sin'] = np.sin(2 * np.pi * df[timestamp_col].dt.dayofweek / 7)
    df['day_cos'] = np.cos(2 * np.pi * df[timestamp_col].dt.dayofweek / 7)


    df['is_weekend'] = df[timestamp_col].dt.dayofweek.isin([5, 6]).astype(int)

    df['is_heating_season'] = df[timestamp_col].dt.month.isin([11, 12, 1, 2, 3]).astype(int)

    return df

In [24]:
forecast_df['timestamp'] = pd.to_datetime(forecast_df['timestamp'],utc=True)
forecast_df = extract_time_features(forecast_df)
forecast_df

,timestamp,sensorId,lat,lon,temperature,humidity,wind_speed,pressure,neighbor1_humidity,neighbor2_humidity,...,neighbor5_wind_speed,season,hour_sin,hour_cos,month_sin,month_cos,day_sin,day_cos,is_weekend,is_heating_season
0,2025-11-09 16:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,12.300,89.000000,2.200000,948.50000,86.0,86.0,...,1.8,autumn,-0.866025,-5.000000e-01,-0.866025,0.5,-0.781831,0.62349,1,1
1,2025-11-09 17:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,12.000,90.000000,3.600000,948.40000,92.0,92.0,...,1.3,autumn,-0.965926,-2.588190e-01,-0.866025,0.5,-0.781831,0.62349,1,1
2,2025-11-09 18:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,11.600,92.000000,2.200000,947.90000,95.0,95.0,...,2.2,autumn,-1.000000,-1.836970e-16,-0.866025,0.5,-0.781831,0.62349,1,1
3,2025-11-09 19:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,11.400,94.000000,1.800000,948.20000,95.0,95.0,...,1.8,autumn,-0.965926,2.588190e-01,-0.866025,0.5,-0.781831,0.62349,1,1
4,2025-11-09 20:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,11.300,95.000000,2.800000,947.80000,94.0,94.0,...,0.7,autumn,-0.866025,5.000000e-01,-0.866025,0.5,-0.781831,0.62349,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501079,2026-03-01 17:00:00+00:00,sensor_dev_8550_72,42.000000,21.380000,5.438,71.722170,1.548418,995.15894,NaN,NaN,...,NaN,spring,-0.965926,-2.588190e-01,0.866025,0.5,-0.781831,0.62349,1,1
501080,2026-03-01 18:00:00+00:00,sensor_dev_8550_72,42.000000,21.380000,7.638,66.715480,1.310420,995.11540,NaN,NaN,...,NaN,spring,-1.000000,-1.836970e-16,0.866025,0.5,-0.781831,0.62349,1,1
501081,2026-03-01 19:00:00+00:00,sensor_dev_8550_72,42.000000,21.380000,6.788,69.456400,2.305125,995.31110,NaN,NaN,...,NaN,spring,-0.965926,2.588190e-01,0.866025,0.5,-0.781831,0.62349,1,1
501082,2026-03-01 20:00:00+00:00,sensor_dev_8550_72,42.000000,21.380000,6.338,68.128060,1.913426,995.45435,NaN,NaN,...,NaN,spring,-0.866025,5.000000e-01,0.866025,0.5,-0.781831,0.62349,1,1


In [25]:
start = pd.Timestamp("2025-11-09 16:00:00",tz="UTC")
end = pd.Timestamp("2025-12-01 00:00:00",tz="UTC")

filtered_df = forecast_df[
    (forecast_df["timestamp"] >= start) &
    (forecast_df["timestamp"] < end)
]

In [26]:
filtered_df

,timestamp,sensorId,lat,lon,temperature,humidity,wind_speed,pressure,neighbor1_humidity,neighbor2_humidity,...,neighbor5_wind_speed,season,hour_sin,hour_cos,month_sin,month_cos,day_sin,day_cos,is_weekend,is_heating_season
0,2025-11-09 16:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,12.300,89.000000,2.200000,948.50000,86.0,86.0,...,1.8,autumn,-0.866025,-5.000000e-01,-0.866025,0.5,-0.781831,0.62349,1,1
1,2025-11-09 17:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,12.000,90.000000,3.600000,948.40000,92.0,92.0,...,1.3,autumn,-0.965926,-2.588190e-01,-0.866025,0.5,-0.781831,0.62349,1,1
2,2025-11-09 18:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,11.600,92.000000,2.200000,947.90000,95.0,95.0,...,2.2,autumn,-1.000000,-1.836970e-16,-0.866025,0.5,-0.781831,0.62349,1,1
3,2025-11-09 19:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,11.400,94.000000,1.800000,948.20000,95.0,95.0,...,1.8,autumn,-0.965926,2.588190e-01,-0.866025,0.5,-0.781831,0.62349,1,1
4,2025-11-09 20:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,11.300,95.000000,2.800000,947.80000,94.0,94.0,...,0.7,autumn,-0.866025,5.000000e-01,-0.866025,0.5,-0.781831,0.62349,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
498897,2025-11-30 19:00:00+00:00,sensor_dev_8550_72,42.000000,21.380000,4.700,73.000000,2.800000,985.90000,NaN,NaN,...,NaN,autumn,-0.965926,2.588190e-01,-0.866025,0.5,-0.781831,0.62349,1,1
498898,2025-11-30 20:00:00+00:00,sensor_dev_8550_72,42.000000,21.380000,4.300,78.000000,3.300000,986.20000,NaN,NaN,...,NaN,autumn,-0.866025,5.000000e-01,-0.866025,0.5,-0.781831,0.62349,1,1
498899,2025-11-30 21:00:00+00:00,sensor_dev_8550_72,42.000000,21.380000,4.000,81.000000,2.600000,986.60000,NaN,NaN,...,NaN,autumn,-0.707107,7.071068e-01,-0.866025,0.5,-0.781831,0.62349,1,1
498900,2025-11-30 22:00:00+00:00,sensor_dev_8550_72,42.000000,21.380000,4.638,88.733864,0.763675,986.44727,NaN,NaN,...,NaN,autumn,-0.500000,8.660254e-01,-0.866025,0.5,-0.781831,0.62349,1,1


In [27]:
filtered_df[TIME_COL] = filtered_df[TIME_COL].dt.tz_convert(None)

/tmp/ipykernel_5878/3348525308.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df[TIME_COL] = filtered_df[TIME_COL].dt.tz_convert(None)


In [28]:
future_df = TimeSeriesDataFrame.from_data_frame(
    filtered_df,
    id_column=ID_COL,
    timestamp_column=TIME_COL
)

In [29]:
predictions = predictor.predict(data=train_data,known_covariates=future_df)

Model not specified in predict, will default to the model with the best validation score: Chronos2


In [30]:
print(predictions.head())

                                                              mean       0.1  \
item_id                              timestamp                                 
007f2b03-94e6-47b3-9e3e-44273354acd5 2025-11-09 16:00:00  2.264434  1.344916   
                                     2025-11-09 17:00:00  2.288295  0.931551   
                                     2025-11-09 18:00:00  2.328097  0.572895   
                                     2025-11-09 19:00:00  2.476265  0.605849   
                                     2025-11-09 20:00:00  2.582566  0.385833   

                                                               0.2       0.3  \
item_id                              timestamp                                 
007f2b03-94e6-47b3-9e3e-44273354acd5 2025-11-09 16:00:00  1.695494  1.902782   
                                     2025-11-09 17:00:00  1.472217  1.779918   
                                     2025-11-09 18:00:00  1.287244  1.712869   
                                     20

In [31]:
performance = predictor.evaluate(test_data)

print(performance)

Model not specified in predict, will default to the model with the best validation score: Chronos2


{'MASE': -1.2917759688608086}


In [32]:
test_df = test_data.to_data_frame()
pred_df = predictions.to_data_frame()

In [33]:
merged = test_df.merge(
    pred_df[["mean"]],
    on=["item_id", "timestamp"],
    how="inner"
)

In [34]:
merged.columns

Index(['humidity', 'pm10', 'pm25', 'pressure', 'temperature', 'wind_speed',
       'neighbor1_humidity', 'neighbor2_humidity', 'neighbor3_humidity',
       'neighbor4_humidity', 'neighbor5_humidity', 'neighbor1_pressure',
       'neighbor2_pressure', 'neighbor3_pressure', 'neighbor4_pressure',
       'neighbor5_pressure', 'neighbor1_temperature', 'neighbor2_temperature',
       'neighbor3_temperature', 'neighbor4_temperature',
       'neighbor5_temperature', 'neighbor1_wind_speed', 'neighbor2_wind_speed',
       'neighbor3_wind_speed', 'neighbor4_wind_speed', 'neighbor5_wind_speed',
       'season', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'day_sin',
       'day_cos', 'is_weekend', 'is_heating_season', 'mean'],
      dtype='object')

In [35]:
merged = merged.rename(columns={"mean": "predicted"})

In [36]:
merged

humidity      pm10  \
item_id                              timestamp                                 
007f2b03-94e6-47b3-9e3e-44273354acd5 2025-11-09 16:00:00     84.00  5.750000   
                                     2025-11-09 17:00:00     84.25  4.250000   
                                     2025-11-09 18:00:00     85.00  3.333333   
                                     2025-11-09 19:00:00     86.25  3.750000   
                                     2025-11-09 20:00:00     87.00  3.250000   
...                                                            ...       ...   
sensor_dev_84941_208                 2025-11-30 19:00:00     83.00       NaN   
                                     2025-11-30 20:00:00     85.50       NaN   
                                     2025-11-30 21:00:00     88.00       NaN   
                                     2025-11-30 22:00:00     89.50       NaN   
                                     2025-11-30 23:00:00     89.00       NaN   

                                                          pm25    pressure  \
item_id                              timestamp                               
007f2b03-94e6-47b3-9e3e-44273354acd5 2025-11-09 16:00:00  3.50  945.000000   
                                     2025-11-09 17:00:00  2.00  945.000000   
                                     2025-11-09 18:00:00  2.00  945.000000   
                                     2025-11-09 19:00:00  1.75  945.000000   
                                     2025-11-09 20:00:00  2.25  945.000000   
...                                                        ...         ...   
sensor_dev_84941_208                 2025-11-30 19:00:00   NaN  981.371212   
                                     2025-11-30 20:00:00   NaN  981.377604   
                                     2025-11-30 21:00:00   NaN  981.609375   
                                     2025-11-30 22:00:00   NaN  981.594086   
                                     2025-11-30 23:00:00   NaN  981.567204   

                                                          temperature  \
item_id                              timestamp                          
007f2b03-94e6-47b3-9e3e-44273354acd5 2025-11-09 16:00:00        13.25   
                                     2025-11-09 17:00:00        13.00   
                                     2025-11-09 18:00:00        13.00   
                                     2025-11-09 19:00:00        13.00   
                                     2025-11-09 20:00:00        13.00   
...                                                               ...   
sensor_dev_84941_208                 2025-11-30 19:00:00         7.00   
                                     2025-11-30 20:00:00         6.75   
                                     2025-11-30 21:00:00         6.00   
                                     2025-11-30 22:00:00         6.00   
                                     2025-11-30 23:00:00         6.00   

                                                          wind_speed  \
item_id                              timestamp                         
007f2b03-94e6-47b3-9e3e-44273354acd5 2025-11-09 16:00:00    4.311284   
                                     2025-11-09 17:00:00    2.845667   
                                     2025-11-09 18:00:00    5.011445   
                                     2025-11-09 19:00:00    4.390272   
                                     2025-11-09 20:00:00    3.476920   
...                                                              ...   
sensor_dev_84941_208                 2025-11-30 19:00:00    0.663094   
                                     2025-11-30 20:00:00    1.077217   
                                     2025-11-30 21:00:00    0.762645   
                                     2025-11-30 22:00:00    0.961617   
                                     2025-11-30 23:00:00    0.839303   

                                                          neighbor1_humidity  \
item_id                            

In [ ]:

from pathlib import Path
import sqlite3

DB_PATH = Path("../data/skopje.db")
if not DB_PATH.exists():
    DB_PATH = Path("data/skopje.db")

if not DB_PATH.exists():
    raise FileNotFoundError("Could not find data/skopje.db. Run the notebook from offline-Phase or the project root.")

CITY = "Skopje"
MODEL_VERSION = f"chronos2_{TARGET}_skopje_offline_test_{PREDICTION_LENGTH}h"
MODEL_TYPE = 'fine_tuned'
train_df = (
    train_data.reset_index().rename(
        columns={
            "item_id": "sensor_id",
            TARGET: "actual_value"
        }
    )
)
train_df['predicted_value'] = np.nan
test_df = (
    merged.reset_index()
    .rename(columns={
        "item_id": "sensor_id",
        TARGET: "actual_value",
        "predicted": "predicted_value",
    })
)
offline_results = pd.concat([train_df,test_df],ignore_index=True).sort_values(["sensor_id", "timestamp"])
offline_results = offline_results[["sensor_id", "timestamp", "actual_value", "predicted_value"]].copy()
offline_results["city"] = CITY
offline_results["pollutant"] = TARGET
offline_results["model_version"] = MODEL_VERSION
offline_results["model_type"] = MODEL_TYPE
offline_results["timestamp"] = pd.to_datetime(offline_results["timestamp"]).dt.strftime("%Y-%m-%d %H:%M:%S")
offline_results = offline_results[["city", "sensor_id", "timestamp", "pollutant", "actual_value", "predicted_value", "model_version","model_type"]]

records = list(offline_results.itertuples(index=False, name=None))

with sqlite3.connect(DB_PATH) as conn:
    conn.execute("""
        CREATE TABLE IF NOT EXISTS offline_test_results (
            city TEXT NOT NULL,
            sensor_id TEXT NOT NULL,
            timestamp TEXT NOT NULL,
            pollutant TEXT NOT NULL,
            actual_value REAL,
            predicted_value REAL,
            model_version TEXT NOT NULL,
            model_type TEXT NOT NULL,
            PRIMARY KEY (city, sensor_id, timestamp, pollutant, model_version)
        )
    """)
    # 2. Add Secondary Indexes for fast querying
    # Fast filtering by time range (e.g. WHERE timestamp >= '2025-11-09')
    conn.execute("""
        CREATE INDEX IF NOT EXISTS idx_offline_timestamp
        ON offline_test_results (timestamp)
    """)

    # Fast filtering by sensor + timestamp queries
    conn.execute("""
        CREATE INDEX IF NOT EXISTS idx_offline_sensor_time
        ON offline_test_results (sensor_id, timestamp)
    """)

    # Fast model performance evaluations (e.g. comparing model versions for a pollutant)
    conn.execute("""
        CREATE INDEX IF NOT EXISTS idx_offline_model_eval
        ON offline_test_results (model_version, pollutant)
    """)
    conn.executemany("""
        INSERT OR REPLACE INTO offline_test_results (
            city, sensor_id, timestamp, pollutant, actual_value, predicted_value, model_version,model_type
        ) VALUES (?, ?, ?, ?, ?, ?, ?,?)
    """, records)

print(f"Saved {len(records)} {CITY} {TARGET} offline test rows to {DB_PATH}")

/tmp/ipykernel_356328/3114147318.py:31: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  offline_results = pd.concat([train_df,test_df],ignore_index=True).sort_values(["sensor_id", "timestamp"])


Saved 192984 Bitola pm25 offline test rows to ../data/skopje.db


In [37]:
merged.dropna(inplace=True)

In [38]:
y_true = merged["pm25"]
y_pred = merged["predicted"]

r2_global = r2_score(y_true, y_pred)
rmse_global = np.sqrt(mean_squared_error(y_true, y_pred))

print("Global R2:", r2_global)
print("Global RMSE:", rmse_global)

Global R2: 0.18721359026617201
Global RMSE: 13.152659648706678


In [39]:
per_sensor = merged.groupby("item_id").apply(
    lambda df: pd.Series({
        "r2": r2_score(df["pm25"], df["predicted"]),
        "rmse": np.sqrt(mean_squared_error(df["pm25"], df["predicted"]))
    })
)

print(per_sensor)

                                             r2       rmse
item_id                                                   
007f2b03-94e6-47b3-9e3e-44273354acd5  -0.034573   2.173253
01cf1cec-bf2d-41b3-8cd5-e8bd720f01b4  -0.799672  11.250481
0a058579-12c9-47be-971b-607198002d3b   0.032452   9.079144
0f10deea-03bc-4a47-ae87-85442140467c  -0.031481   5.195535
1000                                  -0.044473  16.262197
1001                                   0.166357  18.962974
1002                                   0.081631  16.746947
1003                                   0.099251  18.379457
1004                                 -17.193193  20.682982
10661e7e-e9c5-4e67-a5fa-06cdf6f3e76d  -0.019830   8.540234
11888f3a-bc5e-4a0c-9f27-702984decedf   0.153687  13.363600
1286fb13-a4de-44cd-a390-4117fcddf1a9  -0.191228   7.475234
1a2af884-336b-427d-9b37-fe332557539f   0.118090  20.180012
200cdb67-8dc5-4dcf-ac62-748db636e04e   0.168304  19.366695
24eaebc2-ca62-49ff-8b22-880bc131b69f  -0.069215   9.1104

In [40]:
predictor.path

'/content/AutogluonModels/ag-20260803_110505'

In [ ]:
import shutil

shutil.copytree(
    predictor.path,
    "chronos2_model_pm25_skopje"
)

'chronos2_model_pm25_bitola'